#  Modelagem Preditiva e Avaliação

Nesta etapa, estruturamos o modelo preditivo utilizando o **Random Forest Regressor**. A escolha desse algoritmo como modelo primário é estratégica devido às suas propriedades estruturais:

* **Robustez a Overfitting:** Ao construir múltiplas árvores de decisão independentes e realizar a média de suas predições, o algoritmo mitiga a alta variância característica de árvores isoladas.
* **Flexibilidade Geométrica:** É capaz de capturar relações não-lineares complexas e não exige que os dados numéricos sejam escalonados ou normalizados previamente.
* **Resiliência a Ruídos:** Lida de forma estável com pequenas anomalias e outliers no espaço vetorial.

Iniciaremos estabelecendo um modelo base (*baseline*) para, em seguida, otimizar seu desempenho através de um ajuste rigoroso de hiperparâmetros.

In [10]:
# Importando bibliotecas dessa análise
import os
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from scipy import stats
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Configuração de estilo para os gráficos 
sns.set_theme(style="whitegrid")

# Ignorar avisos 
import warnings
warnings.filterwarnings('ignore')

# Definindo o diretório 
caminho = os.getcwd()

# Carregando os datasets
df_treino = pd.read_csv(os.path.join(caminho,"csv_gerados", "df_treino_final.csv"))
df_teste = pd.read_csv(os.path.join(caminho,"csv_gerados", "df_teste_final.csv"))
teste_ids = pd.read_csv(os.path.join(caminho,"csv_gerados", "test_ids.csv"))

## 4.1 Modelo Base (*Baseline*) e Análise de Métricas


O objetivo desse modelo base é estabelecer um referencial mais simples de performace do algorítmo. Os futuros refinamentos (com hiperparâmetros) será justificado futuramente.


In [11]:

# Separação de Features (X) e Alvo (y)
X = df_treino.drop(columns=['SalePrice'])
y = df_treino['SalePrice']

# Divisão do conjunto em Treino e Validação
seed = 42
X_treino, X_validacao, y_treino, y_validacao = train_test_split(
    X, y, test_size=0.2, random_state=seed
)

# Treinamento do Modelo Base
print("Treinando o Modelo Base...")
modelo_base = RandomForestRegressor(n_estimators=100, random_state=seed, n_jobs=-1)
modelo_base.fit(X_treino, y_treino)

# Previsão e Avaliação Interna do Modelo Base
prev_base = modelo_base.predict(X_validacao)

# Índices e métricas
print("=== Avaliação do Modelo Base ===")
print(f"MAE: US$ {mean_absolute_error(y_validacao, prev_base):,.2f}")
print(f"RMSE: US$ {np.sqrt(mean_squared_error(y_validacao, prev_base)):,.2f}")
print(f"MAPE: {np.mean(np.abs((y_validacao - prev_base) / y_validacao)) * 100:.2f}%")
print(f"R²: {r2_score(y_validacao, prev_base):.4f}")

Treinando o Modelo Base...
=== Avaliação do Modelo Base ===
MAE: US$ 16,845.65
RMSE: US$ 24,691.26
MAPE: 10.46%
R²: 0.8896


> **Análise das Métricas (Baseline):**
> 
> * **R² (Poder de Explicação) - 0.8896:** Um resultado inicial excelente. O modelo consegue explicar praticamente **90%** de toda a variação dos preços dos imóveis.
> * **MAPE (Erro Percentual Médio) - 10.46%:** Na média, as previsões se desviam cerca de 10% do valor real de venda. No mercado imobiliário, essa é uma boa margem de segurança para estimativas automáticas.
> * **MAE (Erro Médio Absoluto) - US$ 16.845,65:** Em termos financeiros e práticos, o modelo erra (para mais ou para menos) em torno de 17 mil dólares por casa.
> * **RMSE (Raiz do Erro Quadrático Médio) - US$ 24.691,26:** Como o RMSE penaliza erros grandes de forma quadrática, o fato de ele ser sensivelmente maior que o MAE indica que o modelo ainda comete alguns erros em precificações de casos pontuais (possivelmente casas mais caras e atípicas )

## 4.2 Otimização de Hiperparâmetros 

Embora o modelo base tenha apresentado uma performance robusta, algoritmos como o *Random Forest* possuem dezenas de hiperparâmetros que controlam o nível de complexidade e o crescimento das árvores de decisão. 

Para buscar a configuração ideal, utilizaremos o **`RandomizedSearchCV`**. Ele fará uma amostragem aleatória dentro de um espaço de busca (Grid) definido por nós. 

Para garantir a robustez dessa busca e evitar que o modelo memorize os dados (*overfitting*), acoplamos um processo de **Validação Cruzada com 5 folds (`cv=5`)**. O algoritmo treinará e avaliará o modelo 5 vezes para cada uma das 100 combinações sorteadas (`n_iter=100`), focando em minimizar o nosso Erro Médio Absoluto (MAE) (`scoring='neg_mean_absolute_error'`).

In [12]:
# Definição do espaço de hiperparâmetros (Grid)
parametros_rf = {
    'n_estimators': [100, 300, 500, 800],
    'max_depth': [None, 10, 20, 30],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'max_features': ['sqrt', 'log2', 1.0] 
}

# Configuração da Busca Aleatorizada (Cross-Validation com 5 folds)

rf_random = RandomizedSearchCV(
    estimator=RandomForestRegressor(random_state=seed),
    param_distributions=parametros_rf,
    n_iter=100, 
    cv=5, 
    scoring='neg_mean_absolute_error', # Foca em minimizar o MAE
    n_jobs=-1, 
    random_state=seed,
    verbose=1
)

print("Iniciando a busca de hiperparâmetros...")
rf_random.fit(X_treino, y_treino)

# Extraindo o modelo otimizado e seus parâmetros
modelo_otimizado = rf_random.best_estimator_

print(f"\nMelhores hiperparâmetros encontrados:\n{rf_random.best_params_}\n")

# Previsão e Avaliação do Modelo Otimizado
prev_otimizada = modelo_otimizado.predict(X_validacao)

print("=== Avaliação do Modelo Otimizado ===")
print(f"MAE: US$ {mean_absolute_error(y_validacao, prev_otimizada):,.2f}")
print(f"RMSE: US$ {np.sqrt(mean_squared_error(y_validacao, prev_otimizada)):,.2f}")
print(f"MAPE: {np.mean(np.abs((y_validacao - prev_otimizada) / y_validacao)) * 100:.2f}%")
print(f"R²: {r2_score(y_validacao, prev_otimizada):.4f}")

Iniciando a busca de hiperparâmetros...
Fitting 5 folds for each of 100 candidates, totalling 500 fits

Melhores hiperparâmetros encontrados:
{'n_estimators': 300, 'min_samples_split': 2, 'min_samples_leaf': 1, 'max_features': 'sqrt', 'max_depth': 30}

=== Avaliação do Modelo Otimizado ===
MAE: US$ 15,685.95
RMSE: US$ 22,237.59
MAPE: 10.15%
R²: 0.9105


> **Análise do Modelo Otimizado (Hiperparâmetros):**
> 
> A busca testou diversas combinações para encontrar a melhor configuração possível para o algoritmo. O resultado foi um modelo matematicamente superior ao padrão, entregando **melhorias expressivas e relevantes na prática**.
> 
> * **MAE (Erro Médio):** Caiu de US$ 16.845,65 para **US$ 15.685,95**. Conseguimos reduzir o erro médio de previsão por imóvel em mais de 1.100 dólares.
> * **RMSE (Erro em Extremos):** Reduzido de US$ 24.691,26 para **US$ 22.237,59**. O modelo otimizado lida significativamente melhor com as casas que fogem do padrão (muito caras ou muito baratas), estancando os grandes erros (outliers) em quase 2.500 dólares.
> * **R² (Poder de Explicação):** Passamos de 88,96% para **0.9105** (91,05%). Conseguimos romper a barreira dos 90% com folga, mostrando um salto expressivo na capacidade do modelo de entender os dados.
> * **MAPE (Erro Percentual):** Caiu de 10.46% para **10.15%**, aproximando a taxa geral de erro da excelente marca de um único dígito.
> 
> **Conclusão Prática:**
> De maneira razoável, a otimização computacional melhorou o modelo com um aumento de precisão na faixa de 1000$, o que é um bom valor, porém não possui grande representatividade num contexto imobiliário. Entretanto, esse esforço é importante para obter uma precisão fina no modelo, em especial, para o contexto de competição


## 4.3 Previsões Finais e Arquivo de Submissão

Com o modelo otimizado definido, o último passo é treiná-lo utilizando a totalidade dos dados de treino disponíveis (`X` e `y`), sem reter dados para validação. Isso garante que o algoritmo alcance seu potencial máximo de aprendizado.

Em seguida, realizamos as predições no conjunto de teste (`df_teste`) e estruturamos o resultado no formato exigido pelo Kaggle (colunas `Id` e `SalePrice`), exportando-o como um arquivo `.csv` para enviar a competição.


In [15]:
# Treinando o modelo otimizado com toda a base de treino
print("Treinando o modelo final com 100% dos dados de treino...")
modelo_otimizado.fit(X, y)

# Calculando as previsões
print("Gerando as previsões para o conjunto de teste...")
previsoes_finais = modelo_otimizado.predict(df_teste)

# Criando o DataFrame no formato exigido pelo Kaggle
submissao = pd.DataFrame({
    'Id': teste_ids['Id'],
    'SalePrice': previsoes_finais
})

# Exportando o arquivo CSV final
caminho_submissao = os.path.join(caminho,"csv_gerados","submission.csv")
submissao.to_csv(caminho_submissao, index=False)

print(f"\nSucesso! Arquivo 'submission.csv' gerado com {len(submissao)} linhas.")

Treinando o modelo final com 100% dos dados de treino...
Gerando as previsões para o conjunto de teste...

Sucesso! Arquivo 'submission.csv' gerado com 1459 linhas.
